In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mnn_torch import paths

# QUICK controls only the *recompute fallback*: when a committed grid is absent under
# data/results/, the cell trains a tiny in-kernel version (1 epoch, 1 seed, 2 fault
# probs, 512-example subset) via mnn_torch.training so the notebook still renders a
# figure -- clearly labelled a PREVIEW (near-chance, high-variance). When the grid IS
# present it is always replayed regardless of this flag. Heavy publication sweeps are
# documented as `python -m mnn_torch.training --gate --full` (last cell).
QUICK = False

TEAL, RED, BLUE, GREY, AMBER = "#3aa07a", "#c0392b", "#2f4b8f", "#9aa6b2", "#e0a93b"
INK = "#2b2b2b"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _have(name):
    return (paths.results_dir() / name).exists()

print("data/results:", paths.results_dir())

In [ ]:
grid = None
if _have("gate_sweep.npy"):
    grid = paths.load_result("gate_sweep.npy"); src = "gate_sweep.npy (committed)"
elif _have("gate_sweep_results.npy"):
    grid = paths.load_result("gate_sweep_results.npy"); src = "gate_sweep_results.npy (committed quick grid)"
elif QUICK:
    from mnn_torch.training import gate_homeostasis_sweep, precompute_device_params
    from mnn_torch.data import ensure_mnist
    ensure_mnist()  # download MNIST once into paths.data_dir()
    grid = gate_homeostasis_sweep(
        dev_params=precompute_device_params(), seeds=(0,), probs=(0.0, 0.2, 0.4),
        polarities=("on", "off"), disturb_mode="device_fixed", epochs=1, num_steps=5,
        train_subset=512, test_subset=512, device="cpu", workers=1, verbose=False)
    src = "QUICK preview (1 epoch, 1 seed, 512-subset -- near chance, not the publication grid)"
else:
    raise FileNotFoundError("no committed gate grid; set QUICK=True to preview")

probs = np.asarray(grid["probs"], float)
pols = list(grid["polarities"])
acc = grid["acc"]
POL_COL = {"on": TEAL, "off": RED, "split": BLUE}
POL_LAB = {"on": "stuck-on (hyperactive)", "off": "stuck-off (dead, control)", "split": "stuck split"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for pol in pols:
    col = POL_COL.get(pol, "#444")
    nh = np.asarray(acc[f"{pol}|no"], float).mean(0)
    h = np.asarray(acc[f"{pol}|homeo"], float).mean(0)
    ax1.plot(probs, nh, "--o", color=col, alpha=0.55, ms=4, label=f"{POL_LAB.get(pol, pol)}: no homeo")
    ax1.plot(probs, h, "-o", color=col, ms=4, label=f"{POL_LAB.get(pol, pol)}: + homeo")
    rec_g = np.asarray(acc[f"{pol}|homeo"], float) - np.asarray(acc[f"{pol}|no"], float)
    rec_m = rec_g.mean(0)
    cells = grid["summary"][pol]["cells"]
    lo = np.array([c["ci_lo"] for c in cells]); hi = np.array([c["ci_hi"] for c in cells])
    ax2.plot(probs, rec_m, "-o", color=col, ms=4, label=POL_LAB.get(pol, pol))
    ax2.fill_between(probs, lo, hi, color=col, alpha=0.15)
ax1.axhline(10.0, ls=":", lw=1, color="0.5", label="chance (10%)")
ax1.set_xlabel("device-failure probability $p$"); ax1.set_ylabel("MNIST test accuracy (%)")
ax1.set_title("(a) accuracy vs fault severity", fontsize=10)
ax1.legend(frameon=False, fontsize=7.5); _clean(ax1)
ax2.axhline(0.0, ls="-", lw=0.8, color="0.6")
ax2.set_xlabel("device-failure probability $p$")
ax2.set_ylabel(r"recovery (homeo $-$ no-homeo)  [pp]")
ax2.set_title("(b) homeostatic recovery vs severity", fontsize=10)
ax2.legend(frameon=False, fontsize=8); _clean(ax2)
fig.suptitle(f"Homeostatic fault recovery  [{src}]", fontsize=11)
fig.tight_layout(); plt.show()

for pol in pols:
    s = grid["summary"][pol]; pk = s["peak"]
    line = f"  stuck-{pol:3s}: ideal effect (p=0) = {s['ideal_effect']:+.2f} pp"
    if pk is not None:
        line += f" ; peak fault recovery = {pk['recovery']:+.2f} pp [{pk['ci_lo']:+.2f},{pk['ci_hi']:+.2f}] at p={pk['p']:.2f}"
    print(line)
mt = grid.get("mechanism_test")
if mt is not None:
    print(f"\n  MECHANISM TEST @ p={mt['p']:.2f}: recovery(on)-recovery(off) = "
          f"{mt['diff']:+.2f} pp [{mt['ci_lo']:+.2f},{mt['ci_hi']:+.2f}] -> {mt['verdict']}")

In [ ]:
dose_dir = paths.results_dir() / "dose"
records = []
if dose_dir.exists() and any(dose_dir.glob("rate_*.json")):
    for fn in sorted(dose_dir.glob("rate_*.json")):
        records.append(json.load(open(fn))); src = f"{dose_dir} (committed)"
elif QUICK:
    from mnn_torch.training import dose_response, precompute_device_params
    from mnn_torch.data import ensure_mnist
    ensure_mnist(); dp = precompute_device_params()
    for rate in (0.0, 0.2, 0.4):
        records.append(dose_response(rate, dev_params=dp, seed=0, epochs=1, num_steps=5,
                                     train_subset=512, test_subset=512, device="cpu", verbose=False))
    src = "QUICK preview (1 epoch, 512-subset -- near chance)"
else:
    raise FileNotFoundError("no committed dose grid; set QUICK=True to preview")

records.sort(key=lambda r: r["rate"])
rates = np.array([r["rate"] for r in records], float)
ideal = next((r["ideal"] for r in records if r["rate"] == 0.0), np.nan)
ideal_h = next((r.get("ideal+homeo", np.nan) for r in records if r["rate"] == 0.0), np.nan)
faulty = np.array([r.get("faulty", r.get("ideal", np.nan)) for r in records], float)
faulty_h = np.array([r.get("faulty+homeo", r.get("ideal+homeo", np.nan)) for r in records], float)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4.2))
axA.plot(rates, faulty, "--o", color=GREY, ms=5, label="faulty (no homeo)")
axA.plot(rates, faulty_h, "-o", color=TEAL, ms=5, label="faulty + homeo")
if np.isfinite(ideal):
    axA.axhline(ideal, ls=":", color=BLUE, lw=1.2, label=f"ideal ({ideal:.1f}%)")
axA.axhline(10.0, ls=":", color="0.5", lw=1, label="chance (10%)")
axA.set_xlabel("device-failure rate"); axA.set_ylabel("MNIST test accuracy (%)")
axA.set_title("(a) dose-response: accuracy vs rate", fontsize=10)
axA.legend(frameon=False, fontsize=8); _clean(axA)
rec = np.array([r.get("recovery", np.nan) for r in records], float)
mfin = np.isfinite(rec)
axB.bar(rates[mfin], rec[mfin], width=max(0.03, 0.6 * np.diff(np.sort(rates)).min() if len(rates) > 1 else 0.06),
        color=[TEAL if v > 0 else GREY for v in rec[mfin]], edgecolor=INK, linewidth=0.6)
axB.axhline(0.0, color=INK, lw=1)
axB.set_xlabel("device-failure rate"); axB.set_ylabel("recovery (homeo $-$ no-homeo) [pp]")
axB.set_title("(b) recovery gap per rate", fontsize=10); _clean(axB)
fig.suptitle(f"Dose-response  [{src}]", fontsize=11)
fig.tight_layout(); plt.show()

for r in records:
    if r["rate"] == 0.0:
        print(f"  rate 0.00 (ref): ideal {r['ideal']:.2f}%  ideal+homeo {r.get('ideal+homeo', float('nan')):.2f}%")
    else:
        print(f"  rate {r['rate']:.2f}: faulty {r['faulty']:.2f}%  +homeo {r['faulty+homeo']:.2f}%  "
              f"recovery {r['recovery']:+.2f} pp")

In [ ]:
iso_dir = paths.results_dir() / "isolate"
recs = []
if iso_dir.exists() and any(iso_dir.glob("*.json")):
    for fn in sorted(iso_dir.glob("*.json")):
        recs.append(json.load(open(fn))); src = f"{iso_dir} (committed)"
elif QUICK:
    from mnn_torch.training import isolate_collapse, precompute_device_params
    from mnn_torch.data import ensure_mnist
    ensure_mnist(); dp = precompute_device_params()
    for mp, mode in (("pf", "device"), ("pf", "fixed"), ("ohmic", "device"), ("ohmic", "fixed")):
        recs.append(isolate_collapse(mp, mode, dev_params=dp, rate=0.3, seed=0, epochs=1,
                                     num_steps=5, train_subset=512, test_subset=512,
                                     device="cpu", verbose=False))
    src = "QUICK preview (1 epoch, 512-subset -- near chance)"
else:
    raise FileNotFoundError("no committed isolate grid; set QUICK=True to preview")

labels = [f"{r['map']}\n{r['mode']}" for r in recs]
faulty = np.array([r["faulty"] for r in recs], float)
faulty_h = np.array([r["faulty+homeo"] for r in recs], float)
x = np.arange(len(recs)); w = 0.38
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.bar(x - w / 2, faulty, w, color=GREY, edgecolor=INK, linewidth=0.6, label="faulty")
ax.bar(x + w / 2, faulty_h, w, color=TEAL, edgecolor=INK, linewidth=0.6, label="faulty + homeo")
ax.axhline(10.0, ls=":", color="0.5", lw=1, label="chance (10%)")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("MNIST test accuracy (%)")
ax.set_title(f"Isolate the collapse: forward map x disturbance sampler  [{src}]", fontsize=10)
ax.legend(frameon=False, fontsize=8); _clean(ax)
fig.tight_layout(); plt.show()

for r in recs:
    print(f"  {r['map']:5s}/{r['mode']:6s} @ rate {r['rate']:.2f}: "
          f"faulty {r['faulty']:.2f}%  +homeo {r['faulty+homeo']:.2f}%  recovery {r['recovery']:+.2f} pp")

In [ ]:
# Uncomment to launch the full gate sweep as a subprocess (heavy: spawn Pool, minutes):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mnn_torch.training", "--gate", "--full"])
print("see the markdown above for the full-scale commands")